In [20]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [21]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

'08'

In [27]:
template_file = 'report_for_BD_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

'report_for_BD_2026-09-22.xlsx'

In [ ]:
# 1. Daily Badge impression by device     (no need)
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2

In [ ]:
# 2. Daily Brand Page (LMS) Pageview         (no need? = may be is sr1 page view?)
# 進入search頁面後，點擊少鹽少糖食店按鈕
# 17:35:05|| or.search.layer.search| CityID:0;geo:22.2915336%2C114.2081752;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# unique users; pageview: total click count
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = """
SELECT
    date(time) as querydate,
    platform,
    count(1) as count
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE LOWER(EventAction) LIKE '%or.search.layer.search%'
    AND LOWER(EventLabelRaw) LIKE '%sr:lmssr1%'
GROUP BY 1, 2
"""

In [13]:
#少鹽少糖食店SR1 page view

sql = f'''
with sr1 as
  (select '$$$少鹽少糖食店$$$' AS dummy, platform
  from `openrice-production.ORGA.PV_{year}{str_month}*`
  where -- eventcategory in ('Search Related', 'WebEvent')
   (lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'dedicatedpromotionid')) like '13' or 
       lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'amtid')) like '1093' or
       lower(eventdata) like '%hongkong%amenityid=1093%'))

select platform, count(1) from sr1
group by platform
    '''

df_big_query = client.query(sql).result().to_dataframe()

web = df_big_query.query("platform=='mobile' | platform=='desktop'  ").f0_.sum()
app = df_big_query.query("platform=='android' | platform=='ios' | platform=='hms' ").f0_.sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=2, header=None, index=False)

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [14]:
#少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)
    

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_8656\4189330246.py:9: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [24]:
# Check whether Web has event name / lms id

check_sql = f"""
SELECT
    platform,
    EventCategory,
    EventAction,
    EventLabelRaw,
    EventLabel,
    EventData
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE platform IN ('desktop', 'mobile')
  AND (
      LOWER(EventLabelRaw) LIKE '%lms2.%'
      OR LOWER(EventData) LIKE '%lms2.%'
      OR LOWER(EventLabel) LIKE '%lms2.%'
      OR EXISTS (
          SELECT 1
          FROM UNNEST(eventlabel.list) AS item
          WHERE LOWER(item.value) LIKE '%lms2.%'
      )
  )
LIMIT 100
"""

client.query(check_sql).result().to_dataframe()

BadRequest: 400 No matching signature for function LOWER
  Argument types: STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>>>
  Signature: LOWER(STRING)
    Argument 1: Unable to coerce type STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>>> to expected type STRING
  Signature: LOWER(BYTES)
    Argument 1: Unable to coerce type STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>>> to expected type BYTES at [14:10]; reason: invalidQuery, location: query, message: No matching signature for function LOWER
  Argument types: STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>>>
  Signature: LOWER(STRING)
    Argument 1: Unable to coerce type STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>>> to expected type STRING
  Signature: LOWER(BYTES)
    Argument 1: Unable to coerce type STRUCT<list ARRAY<STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>>

In [25]:
sql = f"""
WITH adv_search AS (
    SELECT platform
    FROM `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE
        (
            LOWER(EventAction) = 'or.advsearch.open.filter'
            OR EXISTS (
                SELECT 1
                FROM UNNEST(eventlabel.list) AS item
                WHERE LOWER(item.value) LIKE '%or.advsearch.open.filter%'
            )
        )
        AND (
            LOWER(EventLabelRaw) LIKE '%lms2%'
            OR LOWER(EventData) LIKE '%lms2%'
            OR EXISTS (
                SELECT 1
                FROM UNNEST(eventlabel.list) AS item
                WHERE LOWER(item.value) LIKE '%lms2%'
            )
        )
)

SELECT
    platform,
    COUNT(1) AS count
FROM adv_search
GROUP BY platform
"""

In [28]:
df_big_query_3 = client.query(sql).result().to_dataframe()

web = df_big_query_3.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_3.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)

BadRequest: 400 Field name value does not exist in STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>> at [11:34]; reason: invalidQuery, location: query, message: Field name value does not exist in STRUCT<item STRUCT<List INT64, Param STRING, Value STRING>> at [11:34]

Location: US
Job ID: 911150ea-2a44-4646-b301-fe829d4b6c54


In [ ]:
# ADV Search
# 3. Daily Theme Listing Impressions --> adv search
# 進入少鹽少糖食店頁面後，點擊篩選搜尋按鈕
# 18:08:25|| or.search.adv| CityID:0;geo:22.2915758%2C114.2081138;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.AdvSearch
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = f"""
with adv_search as (
    select '$$$少鹽少糖食店$$$' AS dummy, platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'or.search.adv'
    AND LOWER(EventLabelRaw) LIKE '%lms%')
)

select platform, count(1) as count
from adv_search
group by platform
"""

df_big_query_5 = client.query(sql).result().to_dataframe()

df_big_query_5


# web = df_big_query_5.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
# app = df_big_query_5.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
# temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,platform,count
0,ios,447
1,hms,952
2,android,29538


In [ ]:
check_sql = f"""
SELECT *
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE platform IN ('desktop', 'mobile')
/*
  AND (
      LOWER(EventLabelRaw) LIKE '%search%'
      OR LOWER(EventLabel) LIKE '%search%'
      OR LOWER(EventData) LIKE '%search%'
      OR EXISTS (
          SELECT 1
          FROM UNNEST(eventlabel.list) AS item
          WHERE LOWER(item.value) LIKE '%search%'
      )
  )
  AND (
      LOWER(EventLabelRaw) LIKE '%lms%'
      OR LOWER(EventLabel) LIKE '%lms%'
      OR LOWER(EventData) LIKE '%lms%'
      OR EXISTS (
          SELECT 1
          FROM UNNEST(eventlabel.list) AS item
          WHERE LOWER(item.value) LIKE '%lms%'
      )
  )
*/
ORDER BY time DESC
LIMIT 100
"""

web_lms_records = client.query(check_sql).result().to_dataframe()
web_lms_records



C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,UserId,IP,EventAction,EventCategory,EventData,EventEntity,EventLabel,EventLabelRaw,EventSource,UserAgent,Product,MachineName,CollectTime,Platform
0,212989,1e14b931-74d2-4439-9f53-2939444dffa5,2026-08-31 23:59:59.900000+00:00,,189.104.23.149,,PageView,,,{'list': []},https://tw.openrice.com/zh-tw/changhua-nantou/...,,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,desktop
1,231618,20bc21be-4531-4e04-a2ef-cb0b658ed182,2026-08-31 23:59:59.900000+00:00,,223.122.92.216,,PageView,,,{'list': []},https://www.openrice.com/zh/hongkong/review/--...,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
2,878920,7ce788ac-0316-4f2d-8590-b7f6ced070e8,2026-08-31 23:59:59.800000+00:00,,103.6.179.250,impression.poi,WebEvent,https://www.openrice.com/zh/hongkong/l-moko新世紀...,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;PoiId:2924,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
3,878920,7ce788ac-0316-4f2d-8590-b7f6ced070e8,2026-08-31 23:59:59.800000+00:00,,103.6.179.250,impression.poi,WebEvent,https://www.openrice.com/zh/hongkong/l-moko新世紀...,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;PoiId:580036,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
4,753898,6b0a1896-2956-4f80-869b-05d494e3df6d,2026-08-31 23:59:59.800000+00:00,,9.151.203.86,,PageView,,,{'list': []},https://tw.openrice.com/zh-tw/changhua-nantou/...,,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,desktop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1284439,b659cc3a-094e-497c-b38a-ff1e6442bd16,2026-08-31 23:59:57.600000+00:00,,123.203.93.25,impression.poi,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;PoiId:991616,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
96,319885,2d4ed3b2-b1cb-4965-b1ad-15acd7989449,2026-08-31 23:59:57.500000+00:00,,49.130.129.96,,PageView,,,{'list': []},https://www.openrice.com/zh/hongkong/r-好鍋日子-屯門...,,Mozilla/5.0 (iPhone; CPU iPhone OS 17_3_1 like...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
97,948708,86c366be-f97e-4ca0-bd8c-6a2d722533e9,2026-08-31 23:59:57.500000+00:00,,192.177.49.91,,PageView,,,{'list': []},https://tw.openrice.com/zh/user/2b98151f-c5fa-...,,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,desktop
98,1202758,aac664cb-67c3-43e8-b444-fac62854050b,2026-08-31 23:59:57.500000+00:00,5491e2cc-ca3c-4f6a-a547-5f7953def06b,119.237.75.210,,PageView,,,{'list': []},https://www.openrice.com/zh/hongkong/r-潮味莊-油麻地...,,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,desktop


In [33]:
web_lms_records.to_csv("web_records.csv", index=False)

In [38]:
# 3. Daily Theme Listing Impressions
# 進入少鹽少糖食店頁面後，點擊篩選搜尋按鈕
# 17:37:14|| or.search.layer.search| CityID:0;geo:22.2915161%2C114.2081815;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# 17:37:14|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = f"""
with adv_search as (
    select '$$$少鹽少糖食店$$$' AS dummy, platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'view.sr1.promotion'
    AND LOWER(EventLabelRaw) LIKE '%lms2%')
)

select platform, count(1) as count
from adv_search
group by platform
"""

df_big_query_3 = client.query(sql).result().to_dataframe()

#df_big_query_3


web = df_big_query_3.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_3.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

temp_df

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date,web,app
0,2026-08,208,1799877


In [39]:
df_big_query_3

,platform,count
0,mobile,107
1,ios,1396935
2,android,397540
3,desktop,101
4,hms,5402
